### Identifying the Best Lipsync Performer in RuPaul’s Drag Race (US Franchise)  ###
RuPaul’s Drag Race is a reality competition show where drag queens compete in various challenges—ranging from fashion design and acting to comedy and performance—for the title of "America’s Next Drag Superstar." A key part of the competition is the lipsync battle, where queens must perform in a showdown to avoid elimination. The show celebrates drag artistry, charisma, uniqueness, nerve, and talent. This notebook analyzes lipsync performance across contestants from RuPaul’s Drag Race and RuPaul’s Drag Race All Stars, using match-by-match data scraped from fandom.com. The goal is to determine which queen performs best in lipsync battles, using multiple performance metrics.

In [26]:
import pickle
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from pyvis.network import Network

with open('./data/lipsync_data.pkl', 'rb') as handle:
    lipsync_data = pickle.load(handle).reset_index(drop=True)

This analysis will be using the lipsync_data dataframe, scraped from fandom.com which contains data regarding the nature of the lipsync (Song title and Artist name) as well as identifier information including what season the lipsync occured and who won and lost. 

In [27]:
lipsync_data.head()

,Song,Artist,Season,Winner,Loser,Queens
0,"yes, and?",Ariana Grande,S17,[Acacia_Forgot],[Hormona_Lisa],"[Acacia_Forgot, Hormona_Lisa]"
1,The Way That You Love Me,Paula Abdul,S17,[Joella],[Lucky_Starzzz],"[Joella, Lucky_Starzzz]"
2,Buttons,The Pussycat Dolls feat. Snoop Dogg,S17,[Kori_King],[Joella],"[Kori_King, Joella]"
3,Boogie Wonderland,"Earth, Wind & Fire and The Emotions",S17,[Lydia_B_Kollins],[Arrietty],"[Lydia_B_Kollins, Arrietty]"
4,get him back!,Olivia Rodrigo,S17,[Lana_Ja'Rae],[Hormona_Lisa],"[Lana_Ja'Rae, Hormona_Lisa]"


I will assess a successful lipsyncer in two ways. Firstly, I will conduct a numerical based analysis, assessing featurings such as win rate per queen, quantifying a success metric. Secondly, I will conduct a graph-based analysis, building a directed graph and assessing the relative page rank. The idea of this is to assess queens (nodes) that have arcs coming from other impactful nodes. Contextually, a queen will score highly if they have beaten queens who have beaten other successful queens. Combining these two metrics in a rank analysis will yeild the most successful lipsyncer across the franchise.

In [28]:
queen_explode = lipsync_data.explode('Queens')
lipsync_data_win_loss = queen_explode.apply(lambda x : 1 if x['Queens'] in x['Winner'] else
                                                                        0 if x['Queens'] in x['Loser'] else
                                                                        'ERR', axis=1)
print("Unitentified queen status count:")
lipsync_data_win_loss.value_counts().get('ERR',0)

Unitentified queen status count:


0

In [29]:
queen_explode['Status'] = lipsync_data_win_loss
lipsync_data_win_loss_df = queen_explode[['Queens', 'Status']]

lipsync_data_win_loss_df = lipsync_data_win_loss_df.groupby('Queens')['Status'].agg(['sum','count']).rename(columns={
    'sum' : 'No. Wins',
    'count' : 'No. Lipsyncs',
})

lipsync_data_win_loss_df['No. Losses'] = lipsync_data_win_loss_df['No. Lipsyncs'] - lipsync_data_win_loss_df['No. Wins']
lipsync_data_win_loss_df['Win Prop.'] = round(lipsync_data_win_loss_df['No. Wins'] / lipsync_data_win_loss_df['No. Lipsyncs'],3)
lipsync_data_win_loss_df.sort_values(by=['No. Wins', 'Win Prop.', 'No. Lipsyncs'], ascending=[False, False, True])

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.
Queens,,,,
Roxxxy_Andrews,10,11,1,0.909
Shea_Couleé,7,8,1,0.875
Angeria_Paris_VanMicheals,7,10,3,0.700
Silky_Nutmeg_Ganache,7,10,3,0.700
Jorgeous,7,11,4,0.636
...,...,...,...,...
Arrietty,0,3,3,0.000
Crystal_Envy,0,3,3,0.000
Hormona_Lisa,0,3,3,0.000


An separate argument can be made for queens who have never lost a lipsync. No data-backed evidence can be ascertained to access where their lipsync successes end. This will hopefully be captured in the success metric captured later.

In [30]:
lipsync_data_win_loss_df[lipsync_data_win_loss_df['No. Losses'] == 0].sort_values(by='No. Wins', ascending=False)

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.
Queens,,,,
Symone,6,6,0,1.0
Ginger_Minj,6,6,0,1.0
Alaska,5,5,0,1.0
Peppermint,4,4,0,1.0
Chad_Michaels,4,4,0,1.0
Naomi_Smalls,4,4,0,1.0
Adore_Delano,3,3,0,1.0
Mistress_Isabelle_Brooks,3,3,0,1.0
Pearl,3,3,0,1.0


The success score is derived as follows:
$$ \text{Success Score} = 2 * \text{No. Wins} - \text{No. Losses} $$
This was done as a lipsync win should be awarded and a lipsync loss should be punished. Equally, those with an equal win to loss difference, but a larger win count should yeild a higher success score, hence the weighting on the number of wins.

In [31]:
lipsync_data_win_loss_df['Success Score'] = (2*lipsync_data_win_loss_df['No. Wins'] - lipsync_data_win_loss_df['No. Losses'])
lipsync_data_win_loss_df = lipsync_data_win_loss_df.sort_values(by='Success Score', ascending=False)
lipsync_data_win_loss_df

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.,Success Score
Queens,,,,,
Roxxxy_Andrews,10,11,1,0.909,19
Shea_Couleé,7,8,1,0.875,13
Symone,6,6,0,1.000,12
Ginger_Minj,6,6,0,1.000,12
Eureka!,6,7,1,0.857,11
...,...,...,...,...,...
Hormona_Lisa,0,3,3,0.000,-3
Crystal_Envy,0,3,3,0.000,-3
Katya,1,6,5,0.167,-3


Using these metrics, the following can be deduced.

In [32]:
ls_first = lipsync_data_win_loss_df.iloc[0].name
ls_last = lipsync_data_win_loss_df.iloc[-1].name

print(f"Most successful queen : {ls_first} with {lipsync_data_win_loss_df.loc[ls_first,'No. Wins']} Wins and {lipsync_data_win_loss_df.loc[ls_first,'No. Losses']} Losses")
print(f"Least successful queen : {ls_last} with {lipsync_data_win_loss_df.loc[ls_last,'No. Wins']} Wins and {lipsync_data_win_loss_df.loc[ls_last,'No. Losses']} Losses")

Most successful queen : Roxxxy_Andrews with 10 Wins and 1 Losses
Least successful queen : Nina_West with 0 Wins and 4 Losses


Next, I'm going to do a network analysis which will hopefully give more context to the data above. Each queen is a node and an arc connecting two nodes will represent a lipsync between the two queens.

In [33]:
G = nx.Graph()

for queens in lipsync_data["Queens"]:
    G.add_nodes_from(queens)
    if len(queens) > 1:
        for pair in combinations(queens, 2):
            G.add_edge(*pair)

In [34]:
net = Network(height='750px', width='100%', notebook=True)
net.from_nx(G)
net.show('queen_network.html')

queen_network.html


I will now make the graph directed. This means that the edges will have a direction to them. An arrow will point from Queen A to Queen B if Queen A lost to Queen B in a lipsync! If its a double win, there will be two arcs pointing in opposing directions.

In [35]:
D = nx.MultiDiGraph()

for _, row in lipsync_data.iterrows():
    for winner in row['Winner']:
        for loser in row['Queens']:
            if winner != loser:
                D.add_edge(loser, winner)

In [36]:
page_rank_dict = nx.pagerank(D)
page_rank_df_ranked = pd.DataFrame({'Queens' : page_rank_dict.keys(), 'Page Rank' : page_rank_dict.values()}).set_index('Queens')

lipsync_data_win_loss_df = lipsync_data_win_loss_df.merge(page_rank_df_ranked, on='Queens')
lipsync_data_win_loss_df

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.,Success Score,Page Rank
Queens,,,,,,
Roxxxy_Andrews,10,11,1,0.909,19,0.025549
Shea_Couleé,7,8,1,0.875,13,0.025917
Symone,6,6,0,1.000,12,0.010965
Ginger_Minj,6,6,0,1.000,12,0.022068
Eureka!,6,7,1,0.857,11,0.021066
...,...,...,...,...,...,...
Hormona_Lisa,0,3,3,0.000,-3,0.000795
Crystal_Envy,0,3,3,0.000,-3,0.000795
Katya,1,6,5,0.167,-3,0.004414


In [37]:
final_df = lipsync_data_win_loss_df.copy()
final_df['Rank_Success'] = final_df['Success Score'].rank(ascending=False)
final_df['Rank_PageRank'] = final_df['Page Rank'].rank(ascending=False)
final_df['Final Score'] = (final_df['Rank_Success'] + final_df['Rank_PageRank']) / 2
final_df.sort_values(by='Final Score').head(10)

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.,Success Score,Page Rank,Rank_Success,Rank_PageRank,Final Score
Queens,,,,,,,,,
Roxxxy_Andrews,10,11,1,0.909,19,0.025549,1.0,3.0,2.00
Shea_Couleé,7,8,1,0.875,13,0.025917,2.0,2.0,2.00
Ginger_Minj,6,6,0,1.000,12,0.022068,3.5,5.0,4.25
Eureka!,6,7,1,0.857,11,0.021066,6.5,6.0,6.25
Kameron_Michaels,6,7,1,0.857,11,0.017623,6.5,11.0,8.75
Jinkx_Monsoon,5,7,2,0.714,8,0.027847,18.0,1.0,9.50
Alaska,5,5,0,1.000,10,0.017568,10.5,12.0,11.25
Naomi_Smalls,4,4,0,1.000,8,0.019230,18.0,7.0,12.50
Angeria_Paris_VanMicheals,7,10,3,0.700,11,0.015454,6.5,19.0,12.75


By ranking the scores produced by the two analyses and then taking the average across the two rankings, we get the above top 10 lipsync performers. 

In [52]:
print('The best lipsyncers are ',', '.join(final_df[final_df['Final Score'] == final_df['Final Score'].min()].index))

The best lipsyncers are  Roxxxy_Andrews, Shea_Couleé


In [38]:
final_df.sort_values(by='Final Score').tail(10)

,No. Wins,No. Lipsyncs,No. Losses,Win Prop.,Success Score,Page Rank,Rank_Success,Rank_PageRank,Final Score
Queens,,,,,,,,,
Gia_Gunn,0,2,2,0.0,-2,0.000795,208.5,193.5,201.00
Xunami_Muse,0,2,2,0.0,-2,0.000795,208.5,193.5,201.00
Jasmine_Masters,0,2,2,0.0,-2,0.000795,208.5,193.5,201.00
Mimi_Imfurst,0,2,2,0.0,-2,0.000795,208.5,193.5,201.00
Plasma,0,2,2,0.0,-2,0.000795,208.5,193.5,201.00
Arrietty,0,3,3,0.0,-3,0.000795,220.0,193.5,206.75
Jan,0,3,3,0.0,-3,0.000795,220.0,193.5,206.75
Hormona_Lisa,0,3,3,0.0,-3,0.000795,220.0,193.5,206.75
Crystal_Envy,0,3,3,0.0,-3,0.000795,220.0,193.5,206.75


In [53]:
print('The worst lipsyncers are ',', '.join(final_df[final_df['Final Score'] == final_df['Final Score'].max()].index))

The worst lipsyncers are  Nina_West


### Appendix ###

Before using the inbuilt pagerank method form NetworkX, I tried calculating the pagerank manually using the random surfer model on a directed graph of lipsync outcomes. In this network, each node represents a queen, and each directed edge represents a lipsync victory (from winner to loser). The random surfer model simulates a person randomly navigating this graph, following outgoing edges with a certain probability or teleporting to a random node, which allows the model to assess not just how often a queen wins, but whom she defeats. This method captures influence and prestige, giving higher scores to queens who beat other well-performing contestants. By using PageRank, I aimed to highlight queens whose wins had greater significance within the competitive landscape, not just those who accumulated the most victories.

In [39]:
lipsync_win_dict = {a : [name for name, counts in b.items() for _ in range(len(counts))] for a,b in D.adjacency()}

In [40]:
import random

no_its = 500000

def visit_random_neighbour(queen_node):

    wins = lipsync_win_dict[queen_node]
    no_wins = len(wins)

    if no_wins == 0:
        return False
    else:
        return wins[random.randint(0,no_wins-1)]
    
def return_rand_queen():
    return list(parsed_queen_df.keys())[random.randint(0,len(parsed_queen_df.keys())-1)]
    
parsed_queen_df = {queen : 0 for queen in lipsync_win_dict.keys()}

rand_queen = return_rand_queen()
for _ in range(no_its):
    parsed_queen_df[rand_queen] += 1
    
    if random.random() > 0.8:
        next_rand = return_rand_queen()
    else:
        next_rand = visit_random_neighbour(rand_queen)
    
    if next_rand:
        rand_queen = next_rand
    else:
        rand_queen = list(parsed_queen_df.keys())[random.randint(0,len(parsed_queen_df.keys())-1)]

page_ranks = {scores[0] : scores[1] for scores in sorted(parsed_queen_df.items(), key=lambda x : x[1])}
pd.DataFrame({'Queens' : page_ranks.keys(), 'Score' : page_ranks.values()}).reset_index().rename(columns={'index' : 'PR_Pos'})

,PR_Pos,Queens,Score
0,0,Lineysha_Sparx,455
1,1,Nicole_Paige_Brooks,471
2,2,Hormona_Lisa,480
3,3,Madame_LaQueer,480
4,4,Crystal_Envy,483
...,...,...,...
219,219,Trinity_The_Tuck,9985
220,220,Ginger_Minj,10439
221,221,Roxxxy_Andrews,11246
222,222,Shea_Couleé,11369
